# TrafficSense: QLoRA extraction + intent slots
Project MP2026CSE126. Select **Runtime > Change runtime type > T4 GPU**.
This workflow fine-tunes Qwen/Qwen2.5-3B-Instruct using Unsloth 4-bit QLoRA.
No training has been executed in the desktop delivery. Colab GPU execution is required.
The tiny authored synthetic corpora are a smoke-training starting point, not thesis validation.
Keep indian_traffic_eval.jsonl held out; never train on ingestion fixture: records.
Reference: https://unsloth.ai/docs/get-started/install/google-colab


In [ ]:
# @file: training/TrafficSense_QLoRA_Finetune.ipynb:install
# @exports: Colab dependencies
# @depends_on: Colab T4 GPU
# @purpose: Install GPU training dependencies in the disposable Colab runtime.
get_ipython().run_line_magic("pip", "install -q unsloth trl datasets accelerate bitsandbytes 'pydantic>=2.10,<3'")


## Upload source/data
Upload a ZIP containing TrafficSense/backend/ and TrafficSense/training/.
Exclude .env, API keys, .venv, caches, and model weights. The ZIP stays in your Colab session.


In [ ]:
# @file: training/TrafficSense_QLoRA_Finetune.ipynb:upload
# @exports: PROJECT
# @depends_on: google.colab
# @purpose: Load the user's project source and annotations safely into Colab.
import json, sys, zipfile
from pathlib import Path
from google.colab import files
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
target = Path('/content').resolve()
with zipfile.ZipFile(archive) as package:
    for member in package.namelist():
        assert (target/member).resolve().is_relative_to(target), 'Unsafe archive path'
    package.extractall(target)
PROJECT = Path('/content/TrafficSense')
assert (PROJECT/'backend/schemas/extraction.py').exists(), 'ZIP must contain TrafficSense/backend'
sys.path.insert(0, str(PROJECT))


In [ ]:
# @file: training/TrafficSense_QLoRA_Finetune.ipynb:data
# @exports: dataset, held_out
# @depends_on: Pydantic contracts and authored training annotations
# @purpose: Validate both task targets and exclude held-out evaluation texts.
from datasets import Dataset
from uuid import NAMESPACE_URL, uuid5
from backend.schemas.extraction import ExtractedIncidentSchema
from backend.schemas.assistant import SlotFillingQuery
def load(name):
    return [json.loads(line) for line in (PROJECT/'training/data'/name).read_text().splitlines() if line.strip()]
train = load('indian_traffic_train.jsonl')
held_out = load('indian_traffic_eval.jsonl')
assert not {r['text'].casefold() for r in train} & {r['text'].casefold() for r in held_out}
records = []
for row in train:
    assert not row.get('source_name', '').startswith('fixture:')
    target = {'report_id': str(uuid5(NAMESPACE_URL, row['id'])), 'landmark_tuple': {}, 'summary_reason': row['text']}
    for name, value in row['expected'].items():
        if name.startswith('landmark_tuple.'):
            target['landmark_tuple'][name.split('.', 1)[1]] = value
        else:
            target[name] = value
    target = ExtractedIncidentSchema.model_validate(target).model_dump(mode='json')
    records.append({'task': 'Extract an Indian traffic incident as schema JSON; abstain on missing lane counts.',
                    'input': 'report_id='+target['report_id']+'\n'+row['text'], 'output': json.dumps(target)})
for row in load('assistant_train.jsonl'):
    target = SlotFillingQuery.model_validate(row['output']).model_dump(mode='json')
    records.append({'task': 'Classify intent and fill allowed traffic-query slots as JSON. Never emit SQL.',
                    'input': row['text'], 'output': json.dumps(target)})
dataset = Dataset.from_list(records)
print('Training examples:', len(dataset), 'Held-out incident examples:', len(held_out))


In [ ]:
# @file: training/TrafficSense_QLoRA_Finetune.ipynb:model
# @exports: model, tokenizer
# @depends_on: unsloth, CUDA
# @purpose: Load the requested model in 4-bit mode and attach reproducible LoRA adapters.
from unsloth import FastLanguageModel
import torch
assert torch.cuda.is_available(), 'Select a Colab T4 GPU runtime first'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-3B-Instruct', max_seq_length=2048, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none', use_gradient_checkpointing='unsloth', random_state=42)
def format_row(row):
    messages = [{'role':'system','content':row['task']}, {'role':'user','content':row['input']},
                {'role':'assistant','content':row['output']}]
    return {'text':tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}
dataset = dataset.map(format_row)


In [ ]:
# @file: training/TrafficSense_QLoRA_Finetune.ipynb:train
# @exports: trainer, training_result
# @depends_on: trl, Unsloth model and validated dataset
# @purpose: Run a bounded T4 smoke fine-tune and save adapters and environment provenance.
from trl import SFTTrainer, SFTConfig
from inspect import signature
import subprocess
options = dict(output_dir='/content/trafficsense_adapter', per_device_train_batch_size=1,
    gradient_accumulation_steps=4, max_steps=60, warmup_steps=5, learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5, optim='adamw_8bit', weight_decay=0.01, seed=42, report_to='none',
    dataset_text_field='text')
length_name = 'max_length' if 'max_length' in signature(SFTConfig).parameters else 'max_seq_length'
options[length_name] = 2048
trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=dataset, args=SFTConfig(**options))
training_result = trainer.train()
model.save_pretrained('/content/trafficsense_adapter')
tokenizer.save_pretrained('/content/trafficsense_adapter')
Path('/content/trafficsense_adapter/environment.txt').write_bytes(subprocess.check_output([sys.executable,'-m','pip','freeze']))
Path('/content/trafficsense_adapter/training_metrics.json').write_text(json.dumps(training_result.metrics, indent=2))


## Validate and export
Adapter generation is not a deployment or an accuracy claim. Evaluate held-out data after training,
validate generated JSON with ExtractedIncidentSchema/SlotFillingQuery, and report errors honestly.
Grammar constraints and desktop lane-evidence checks are still required at inference time.


In [ ]:
# @file: training/TrafficSense_QLoRA_Finetune.ipynb:export
# @exports: adapter archive
# @depends_on: completed training and validated schema smoke inference
# @purpose: Verify one structured output and download adapters without publishing to a model hub.
FastLanguageModel.for_inference(model)
sample = records[0]
inputs = tokenizer.apply_chat_template([{'role':'system','content':sample['task']},
    {'role':'user','content':sample['input']}], tokenize=True, add_generation_prompt=True,
    return_tensors='pt').to('cuda')
output = model.generate(input_ids=inputs, max_new_tokens=512, do_sample=False)
generated = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
ExtractedIncidentSchema.model_validate_json(generated)
import shutil
archive = shutil.make_archive('/content/trafficsense_adapter', 'zip', '/content/trafficsense_adapter')
files.download(archive)
